## Задание 1. (1 балл)

Посчитайте частоты для 5-грамм в корпусе lenta.txt. двумя способами:  
1) Текст токенизируется в один сплошной список токенов и нграммы считаются по всем токенам без учета границ предложений
2) Текст разбивается на предложения, а каждое предложение на токены; нграммы собираются с учетом границ предложений 
    
Найдите различия между топ 100 нграммов для первого и второго способов

In [4]:
import os, re, gzip, csv, io, urllib.request
from collections import Counter

try:
    from razdel import tokenize as _razdel_tokenize, sentenize as _razdel_sentenize
    def tokenize(text):
        return [t.text for t in _razdel_tokenize(text)]
    def sentenize(text):
        return [s.text for s in _razdel_sentenize(text)]
    print("Используем razdel")
except Exception as e:
    print("razdel недоступен, используем regex-fallback:", e)
    _tok_re = re.compile(r"\w+|[^\w\s]", re.UNICODE)
    _sent_re = re.compile(r"[^.!?…]+[.!?…]+|[^.!?…]+$")
    def tokenize(text):
        return _tok_re.findall(text)
    def sentenize(text):
        return [s.strip() for s in _sent_re.findall(text) if s.strip()]


LENTA_PATH = "lenta.txt"
LENTA_GZ   = "lenta-ru-news.csv.gz"
URL = "https://github.com/yutkin/Lenta.Ru-News-Dataset/releases/download/v1.0/lenta-ru-news.csv.gz"
MAX_NEWS = 3000   

if not os.path.exists(LENTA_PATH):
    if not os.path.exists(LENTA_GZ):
        print("Скачиваем датасет Lenta.ru")
        urllib.request.urlretrieve(URL, LENTA_GZ)
        print("Готово.")

    print("Парсим csv.gz стандартными средствами")
    texts = []
    with gzip.open(LENTA_GZ, "rt", encoding="utf-8", newline="") as f:
        reader = csv.reader(f)
        header = next(reader, None)  
        for row in reader:
            if len(row) >= 3 and row[2]:
                texts.append(row[2])
            if len(texts) >= MAX_NEWS:
                break

    with open(LENTA_PATH, "w", encoding="utf-8") as f:
        f.write("\n".join(texts))
    print(f"Сохранено {len(texts)} новостей в {LENTA_PATH}")

with open(LENTA_PATH, encoding="utf-8") as f:
    corpus = f.read()

print(f"Размер корпуса: {len(corpus):,} символов")

#Способ 1
tokens_flat = tokenize(corpus)
print(f"Всего токенов (способ 1): {len(tokens_flat):,}")

def get_ngrams(tokens, n):
    return [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]

ngrams_flat = Counter(get_ngrams(tokens_flat, 5))
top100_flat = ngrams_flat.most_common(100)

#Способ 2
sentences = sentenize(corpus)
print(f"Всего предложений: {len(sentences):,}")

ngrams_sent = Counter()
for s in sentences:
    toks = tokenize(s)
    ngrams_sent.update(get_ngrams(toks, 5))

top100_sent = ngrams_sent.most_common(100)

#Различия
set_flat = {ng for ng, _ in top100_flat}
set_sent = {ng for ng, _ in top100_sent}

only_flat = set_flat - set_sent
only_sent = set_sent - set_flat
print(f"\nТолько в способе 1: {len(only_flat)}")
print(f"Только в способе 2: {len(only_sent)}")

print("\nПримеры 5-грамм, встречающихся только в способе 1 (без учёта границ):")
for ng in list(only_flat)[:10]:
    print("  ", " ".join(ng))

print("\nПримеры 5-грамм, встречающихся только в способе 2 (с учётом границ):")
for ng in list(only_sent)[:10]:
    print("  ", " ".join(ng))

print("\nТоп-10 способ 1:")
for ng, cnt in top100_flat[:10]:
    print(f"  {' '.join(ng):<60} {cnt}")

print("\nТоп-10 способ 2:")
for ng, cnt in top100_sent[:10]:
    print(f"  {' '.join(ng):<60} {cnt}")

Используем razdel
Размер корпуса: 3,903,693 символов
Всего токенов (способ 1): 642,839
Всего предложений: 35,387

Только в способе 1: 20
Только в способе 2: 20

Примеры 5-грамм, встречающихся только в способе 1 (без учёта границ):
   . Ранее сообщалось , что
   . Он добавил , что
   . Об этом сообщает РИА
   . Об этом в пятницу
   . По его мнению ,
   . Президент России Владимир Путин
   . Об этом сообщает The
   . По ее словам ,
   . Президент Украины Петр Порошенко
   . Об этом сообщается на

Примеры 5-грамм, встречающихся только в способе 2 (с учётом границ):
   Яны Капу » , которые
   . ру » ) .
   этом сообщается в пресс-релизе ,
   » , — подчеркнул он
   Об этом в понедельник ,
   уголовное дело о нарушении государственной
   Об этом во вторник ,
   в пятницу , 14 декабря
   » , — отметил он
   Об этом сообщает « Интерфакс

Топ-10 способ 1:
  . По его словам ,                                            314
  « Ленты . ру »                                               286
  » , —

## Задание 2. (1 балл)

Найдите какую-то инетересную (по вашему мнению) закономерность на https://books.google.com/ngrams/ для русского языка (с 1990 по 2022)

Вставьте сюда скриншот

In [ ]:
https://drive.google.com/file/d/1RCI4afd89UYe9d0_5J7kwrvKc7Po_7KI/view?usp=drive_link

## Заданиe 3 (1 балл)

В семинаре мы использовали Dice coefficient (`score = 2 * bigram_count/((word_count_a+word_count_b))`) для нахождения коллокаций. 
Но самая известная метрика для этого другая - [PMI](https://en.wikipedia.org/wiki/Pointwise_mutual_information). Имплементируйте скоринг с помощью этой метрики и сравните результаты с Dice на топ 30 биграмах. 
Используйте логарифмы для работы с вероятностями. 

In [3]:
def scorer_pmi(*args):
    try:
        score = ...
    except ZeroDivisionError:
        return 0
    return score

In [4]:
import math

def scorer_pmi(bigram_count, word_count_a, word_count_b, total_count):
    """
    PMI = log2( P(w1,w2) / (P(w1) * P(w2)) )
    Все вероятности считаются как доли от total_count
    """
    try:
        p_ab = bigram_count / total_count
        p_a = word_count_a / total_count
        p_b = word_count_b / total_count
        score = math.log2(p_ab / (p_a * p_b))
    except ZeroDivisionError:
        return 0
    return score

# Задание 4 (2 балл)

Шаг 1: Измените функцию merge_round так чтобы она использовала скоринг, который вы сделаете в задании выше. 
Шаг 2: Подберите параметры (min_count, n_merges, n_rounds) так чтобы получались адекватные результаты
Шаг 3: Сохраните промежуточные словари мержей и напишите функцию, которая их последовательно применяет, чтобы иметь возможности воспроизвести трансформацию на любой новом тексте 

Подберите несколько предложений, на которых функция трансформации будет склеивать хотя бы несколько слов вместе 

In [11]:
import os, re, gzip, csv, math, urllib.request
from collections import Counter

try:
    from razdel import tokenize as _razdel_tokenize, sentenize as _razdel_sentenize
    def tokenize(text):
        return [t.text for t in _razdel_tokenize(text)]
    def sentenize(text):
        return [s.text for s in _razdel_sentenize(text)]
    print("Используем razdel")
except Exception as e:
    print("razdel недоступен, regex-fallback:", e)
    _tok_re = re.compile(r"\w+|[^\w\s]", re.UNICODE)
    _sent_re = re.compile(r"[^.!?…]+[.!?…]+|[^.!?…]+$")
    def tokenize(text):
        return _tok_re.findall(text)
    def sentenize(text):
        return [s.strip() for s in _sent_re.findall(text) if s.strip()]

LENTA_PATH = "lenta.txt"
LENTA_GZ = "lenta-ru-news.csv.gz"
URL = ("https://github.com/yutkin/Lenta.Ru-News-Dataset/releases/"
       "download/v1.0/lenta-ru-news.csv.gz")
MAX_NEWS = 3000

if not os.path.exists(LENTA_PATH):
    if not os.path.exists(LENTA_GZ):
        print("Скачиваем Lenta.ru...")
        urllib.request.urlretrieve(URL, LENTA_GZ)
    texts = []
    with gzip.open(LENTA_GZ, "rt", encoding="utf-8", newline="") as f:
        reader = csv.reader(f)
        next(reader, None)
        for row in reader:
            if len(row) >= 3 and row[2]:
                texts.append(row[2])
            if len(texts) >= MAX_NEWS:
                break
    with open(LENTA_PATH, "w", encoding="utf-8") as f:
        f.write("\n".join(texts))

with open(LENTA_PATH, encoding="utf-8") as f:
    corpus = f.read()
sentences = sentenize(corpus)
print(f"Предложений: {len(sentences):,}")

def scorer_pmi(bigram_count, word_count_a, word_count_b, total_count):
    try:
        p_ab = bigram_count / total_count
        p_a = word_count_a / total_count
        p_b = word_count_b / total_count
        return math.log2(p_ab / (p_a * p_b))
    except (ZeroDivisionError, ValueError):
        return 0.0

def merge_round(corpus, min_count=10, n_merges=50):
    bigram_counts = Counter()
    unigram_counts = Counter()
    total_tokens = 0

    for sent in corpus:
        total_tokens += len(sent)
        for i in range(len(sent) - 1):
            bigram_counts[(sent[i], sent[i + 1])] += 1
        for token in sent:
            unigram_counts[token] += 1

    candidates = {}
    for bg, cnt in bigram_counts.items():
        if cnt >= min_count:
            candidates[bg] = cnt

    scores = {}
    for (w1, w2), cnt in candidates.items():
        scores[(w1, w2)] = scorer_pmi(
            cnt, unigram_counts[w1], unigram_counts[w2], total_tokens
        )

    if not scores:
        print("  нет кандидатов — раунд пропущен")
        return corpus, {}

    best = sorted(scores.items(), key=lambda kv: kv[1], reverse=True)
    best = best[:n_merges]

    merges = {}
    for (w1, w2), _ in best:
        merges[(w1, w2)] = w1 + "_" + w2

    new_corpus = []
    for sent in corpus:
        new_sent = []
        i = 0
        while i < len(sent):
            if i < len(sent) - 1 and (sent[i], sent[i + 1]) in merges:
                new_sent.append(merges[(sent[i], sent[i + 1])])
                i += 2
            else:
                new_sent.append(sent[i])
                i += 1
        new_corpus.append(new_sent)

    return new_corpus, merges

def apply_merges(text, merges_list):
    if isinstance(text, str):
        tokens = tokenize(text)
    else:
        tokens = list(text)

    for merges in merges_list:
        new_tokens = []
        i = 0
        while i < len(tokens):
            if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) in merges:
                new_tokens.append(merges[(tokens[i], tokens[i + 1])])
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens

    return " ".join(tokens)

sample_sentences = sentences[:2000]
corpus_tokens = [tokenize(s) for s in sample_sentences]
print(f"Предложений: {len(corpus_tokens)}, "
      f"токенов: {sum(len(s) for s in corpus_tokens)}")

saved_merged = []
sub_corpus = corpus_tokens
for round_i in range(1, 8):  # 7 раундов — пока не надоест
    sub_corpus, merges = merge_round(sub_corpus, min_count=15, n_merges=40)
    if not merges:
        print(f"Раунд {round_i}: пусто, дальше смысла нет")
        break
    saved_merged.append(merges)
    print(f"Раунд {round_i}: {len(merges)} склеек, "
          f"токенов стало {sum(len(s) for s in sub_corpus)}")

print("\nТоп склеек раунда 1:")
for pair, merged in list(saved_merged[0].items())[:15]:
    print("  ", pair, "->", merged)

if len(saved_merged) >= 2:
    print("\nТоп склеек раунда 2:")
    for pair, merged in list(saved_merged[1].items())[:15]:
        print("  ", pair, "->", merged)

test_text = ("Искусственный интеллект меняет рынок труда в России "
             "Нейросети уже пишут тексты и код, а также помогают врачам "
             "ставить диагнозы. Однако рынок труда реагирует медленно")

print("\nДо мержей:")
print("  ", " ".join(tokenize(test_text)))

print("\nПосле мержей:")
print("  ", apply_merges(test_text, saved_merged))

Используем razdel
Предложений: 35,387
Предложений: 2000, токенов: 36139
Раунд 1: 40 склеек, токенов стало 34900
Раунд 2: 38 склеек, токенов стало 33798
Раунд 3: 4 склеек, токенов стало 33629
  нет кандидатов — раунд пропущен
Раунд 4: пусто, дальше смысла нет

Топ склеек раунда 1:
   ('РИА', 'Новости') -> РИА_Новости
   ('стало', 'известно') -> стало_известно
   ('УПЦ', 'МП') -> УПЦ_МП
   ('православной', 'церкви') -> православной_церкви
   ('со', 'ссылкой') -> со_ссылкой
   ('во', 'время') -> во_время
   ('Об', 'этом') -> Об_этом
   ('14', 'декабря') -> 14_декабря
   ('2018', 'года') -> 2018_года
   ('По', 'данным') -> По_данным
   ('13', 'декабря') -> 13_декабря
   ('При', 'этом') -> При_этом
   ('этом', 'сообщает') -> этом_сообщает
   ('15', 'декабря') -> 15_декабря
   ('По', 'словам') -> По_словам

Топ склеек раунда 2:
   ('субботу_,', '15_декабря') -> субботу_,_15_декабря
   ('в_пятницу', ',_14') -> в_пятницу_,_14
   ('По_его', 'словам') -> По_его_словам
   (',_14', 'декабря') -> ,

# Задание 5 (3 балла)

Напишите краткое эссе (30 - 50 предложений) в свободной форме по одной из этих статей:
1) https://news.microsoft.com/source/2026/09/09/aft-uft-and-microsoft-announce-national-ai-safety-privacy-standard-for-schools-to-protect-students-families-and-educators/ и сам целый документ https://www.aft.org/sites/default/files/media/documents/2026/NAfAI-School_AI_Privacy_Standards.pdf
2) https://bayareacurrent.com/theres-a-lot-of-cleaning-up-poop-a-depot-worker-on-the-human-labor-behind-waymos-autonomous-vehicles/

Эту часть достаточно сложно оценивать формально, но я ожидаю увидеть ваши содержательные рассуждения о написанном в статье.   
Если после прочтения вам ничего не приходит в голову, то вы можете начать со следующих вопросов: Считаете ли вы эту информацию важной/интересной/страшной/скучной/и тп? Что показалось вам наиболее важным? Почему? Считаете ли вы эту статью объективной? Какие пробелы вы видите? По вашему мнению будет ли продолжение у этих историй? В каком направлении они будут развиваться? Считаете ли вы age restriction необходимым для AI моделей? (для первой статьи) Как можно предотвращать/бороться с подобным антисоциальным поведением в автономных автомобилях? (для второй) ... 
Но это не полный и не обязательный список! Вы можете писать о всем, что вы посчитаете нужным. 

Чего точно следует избегать:
1) Не пишите саммари! Ваше эссе должно выражать ваши собственные мысли и добавлять что-то к тексту, а не заменять его.
2) Не гонитесь за полнотой! Можно и нужно опускать детали. Вы даже можете даже сфокусироваться только на каком-то одном аспекте статьи и порассуждать о нем поглубже.
3) Не отвлекайтесь на форматирование! Не используйте нумерованные списки, булет пойнты, ссылки и т.п. Напишите все как один связанный plain текст (можете разбить все на параграфы, но не более того)
4) Не объясняйте термины и понятия, не давайте определений! Считайте, что читатель знает столько же, сколько вы или же что он может погуглить.
5) Не уходите от темы! Реагируйте только на сам текст, а не на общий контекст (схожие события и работы, реакция в сотсетях и т.п). Не сравнивайте с другими статьями от того же Антропика или OpenAI.
6) Не бойтесь выражать мнение! Но либо обосновывайте его, либо описиваете свои ощущения в деталях.
7) Будьте осторожными с bias. Не списывайте все на капитализм, AI buble, AI doomerism, AI accerationalism и тому подобные слишком общие и абстрактные вещи. Не выражайте ваше личное отношение к авторам (антропику), только к содержанию статьи! 



Статья интересна тем, что показывает разрыв между публичным образом автономных автомобилей и тем, как они работают на самом деле. Waymo продаёт идею технологии, которая убирает человека из процесса. Однако из рассказа работника депо видно, что человек никуда не исчез, а стал невидимым. Его вынесли за пределы салона, за пределы рекламного ролика и за пределы разговора о технологиях. 

Наиболее важным мне показалось то, как организован труд этих людей. Работник описывает свою задачу фразой простой фразой: «мы просто выключаем и включаем машину». Получается, что вся сложная система с сенсорами и алгоритмами в критический момент сводится к тому, что человек должен понимать, когда нужно нажать кнопочку. С одной стороны, присутствует некоторая автономия, но при этом сохраняется необходимость в подстраховке, которая должна быть незаметной. Труд работников плохо оплачивается и, по сути, похож на труд "маленького человека". В рамках психологии труд, который является важным и значимым по объекивным причинам, но при этом незаметным для общества, может в долгосрочной перспективе наносить серьезный урон психике людей, выполняющих его. Выстраевается система, в которой невидимость работника является условием существования продукта, причём достаточно значимым. К тому же, что сам формат статьи является попыткой говорить о труде, который по определению не предназначен для публичного обсуждения. Компания строит образ комфортного и незаметного сервиса, а статья показывает, что этот образ держится на людях, чья работа состоит в том, чтобы устранять неполадки до того, как их заметит пассажир. Работники достаточно буднично и привычно описывают вещи, которые для внешнего наблюдателя выглядят как проблема. Надёжность автономных автомобилей представляет собой не только вопрос инженерии, но и вопрос того, сколько людей готовы держать эту надёжность на своих плечах.

Данная статья больше похожа на реальную историю, которая призывает к вниманию к довольно незаметной проблеме. Сама по себе идея и технология не плохая, но присутствует обратная сторона медали. Иногда за автоматизацией стоит ручной труд, и вопрос в том, насколько этот труд виден и насколько хорошо за него платят. Если компания экономит не на технологиях, рекламе или поддержании своего бренда, то она может экономить на людях, которые обслуживают процессы, их условиях труда. В итоге беспилотность оказывается скорее маркетинговой историей, чем описанием того, как всё устроено на самом деле. Неприятно, что чем убедительнее эта история, тем меньше у общества поводов вспоминать о тех, кто остаётся за кадром. 

Как я уже отметила, подобная организация труда встречается, в том числе в других сервисах, которые мы привыкли воспринимать как автоматические. Разница лишь в том, что в случае с Waymo невидимость является принципом, а может даже основной ценностью. Мы получаем ситуацию, когда чем сильнее технология обещает избавить нас от чужого труда, тем тщательнее этот труд ей приходится прятать. Многие современные технологии, которые выглядят гладко снаружи, обычно требуют чьей-то работы внутри, даже если это преподносится как автоматизация. Человеческий труд никуда не исчезает, он просто приобретает другой формат, который выгоднее компании.

Я не думаю, что проблема, о которой идёт речь может быть решена в рамках текущей системы. Можно повысить зарплату, можно улучшить условия, можно заставить компанию предоставлять машины бесплатно, но сама логика невидимого труда остаётся (и не только в рамках этой и похожих компаний). Пока есть спрос на «беспилотное» такси, будет и спрос на людей, которые за ним присматривают. Более того, чем лучше технология маскирует своё несовершенство, тем больше работы перекладывается на тех, кто находится в тени. Я думаю, что здесь стоит искать комплексное решение, для многих профессий такого "невидимого" формата.